# SIH 2026: CCTV Video Quality Diagnostics & Metrics Distribution
This notebook analyzes the multi-dimensional degradation signatures (Brightness, Blur, Contrast, Resolution) across CCTV frames.

In [ ]:
import os
import sys
import cv2
import numpy as np
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.abspath('..'))
from src.video.reader import VideoReader
from src.quality.analyzer import QualityAnalyzer

## 1. Run Quality Diagnostic Pipeline on Sample Video

In [ ]:
sample_video = '../videos/VIRAT_S_010205_04_000545_000576.mp4'
analyzer = QualityAnalyzer()
reader = VideoReader(sample_video)

luminance_vals = []
blur_scores = []
contrast_vals = []
recommendations = []

print(f"Analyzing video: {sample_video}...")
for idx, frame in reader.iter_frames(max_frames=60):
    report = analyzer.analyze_frame(frame)
    luminance_vals.append(report.brightness['mean_luminance'])
    blur_scores.append(report.blur['laplacian_var'])
    contrast_vals.append(report.contrast['rms_contrast'])
    recommendations.extend(report.recommended_enhancements)

reader.release()
print(f"Analyzed {len(luminance_vals)} frames.")

## 2. Quality Metric Trajectories Across Video Frames

In [ ]:
fig, axs = plt.subplots(3, 1, figsize=(12, 8), sharex=True)

axs[0].plot(luminance_vals, color='orange', lw=2)
axs[0].axhline(y=75, color='red', linestyle='--', label='Low-light Threshold (75)')
axs[0].set_ylabel('Mean Luminance')
axs[0].set_title('Luminance Profile (Zero-DCE Trigger)')
axs[0].legend()
axs[0].grid(True, alpha=0.3)

axs[1].plot(blur_scores, color='blue', lw=2)
axs[1].axhline(y=120, color='red', linestyle='--', label='Blur Threshold (120)')
axs[1].set_ylabel('Laplacian Variance')
axs[1].set_title('Sharpness Profile (RVRT Trigger)')
axs[1].legend()
axs[1].grid(True, alpha=0.3)

axs[2].plot(contrast_vals, color='green', lw=2)
axs[2].set_ylabel('RMS Contrast')
axs[2].set_xlabel('Frame Index')
axs[2].set_title('Contrast Profile')
axs[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()